## MAT-1 check from burst continuous 
 read the data with pandas and plot them to check if there was anymovement while deployed

In [ ]:
# The routine to read the MAT-1 dada.csv, average, plots and write a nc


# ------------------------------------------------------------
# Imports

import numpy as np
import xarray as xr
import os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import glob
import gsw
import datetime
import pandas as pd

# End imports
# ------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------
# Definitions

def compute_orientation(df):
    Ax = df["Ax (g)"].to_numpy()
    Ay = df["Ay (g)"].to_numpy()
    Az = df["Az (g)"].to_numpy()
    Mx = df["Mx (mG)"].to_numpy()
    My = df["My (mG)"].to_numpy()
    Mz = df["Mz (mG)"].to_numpy()

    # Pitch and Roll from accelerometer
    pitch = np.arctan2(-Ax, np.sqrt(Ay**2 + Az**2))
    roll = np.arctan2(Ay, Az)

    # Yaw estimate from magnetometer and pitch/roll
    mag_x = Mx * np.cos(pitch) + Mz * np.sin(pitch)
    mag_y = Mx * np.sin(roll) * np.sin(pitch) + My * np.cos(roll) - Mz * np.sin(roll) * np.cos(pitch)
    yaw = np.arctan2(-mag_y, mag_x)

    # Convert radians to degrees
    pitch_deg = np.degrees(pitch)
    roll_deg = np.degrees(roll)
    yaw_deg = np.degrees(yaw)

    df["Pitch"] = pitch_deg
    df["Roll"] = roll_deg + 180
    df["Yaw"] = yaw_deg
    # Optional: wrap angles to 0–360
    # df["Pitch"] = (df["Pitch"] + 360) % 360
    # df["Roll"] = (df["Roll"] + 360) % 360
    df["Yaw"] = (df["Yaw"] + 360) % 360

    return df

# End Definitions
# ------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------
# Start Main
# ------------------------------------------------------------

# Definitions

# naming params 
channels = 'acc'
# sd = r_start_time.strftime("%Y%m%dT%H%M%SZ")
processing_ver = 2
inst_type = 'MAT1'

# goes to the targeted directory where the data file is expected. 
fgen = '/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/data_in'

# folder = fgen+'/rec_202408/BASS3A_PTSUVW_202408/RBRQ3_209968'
list_folders=['rec_202502/BASJAS_PTSUVW_202502/MAT1_2004026']

# Loop on folders to process
for f in list_folders:
    sn = f.split("_")[4]
    sni = int(f.split("_")[4])
    site = f.split("_")[1].split("/")[1].split("BAS")[1]
    print(f"{f} : sn = {sn} , site = {site}")
    folder=fgen+'/'+f
    # os.chdir(folder)
    # os.getcwd()
    filist = sorted(glob.glob(f"{folder}/*_AccelMag.csv"))
    file_in=filist[0]  # incase there's multiple _data file it only looks at the first one. if there is to be multiple file for a deployment, one would have a to look into looping throught the list or something.
    pf_code = f"BAS{site}"
    print('processing ',file_in)
    # read that file into a pandas dataframe
    df = pd.read_csv(file_in , parse_dates=["ISO 8601 Time"])
    ## Define start and end times and select the "in water" part of the dataset (still a pandas dataframe)

# Rename the timestamp column for convenience
df.rename(columns={"ISO 8601 Time": "Time"}, inplace=True)

print(df.head())


In [ ]:
# compute the orientations
df = compute_orientation(df)
df

In [ ]:
time_start = pd.to_datetime("2024-02-16T07:30:00")
# time_end = pd.to_datetime("2025-02-7T03:00:00")
time_end = pd.to_datetime("2024-02-17T03:00:00")
df_filtered = df[(df["Time"] >= time_start) & (df["Time"] <= time_end)]

fig, ax1 = plt.subplots(figsize=(10, 5))

# Left Y-axis for pitch and roll
ax1.plot(df_filtered["Time"], df_filtered["Pitch"], label="Pitch", color="tab:blue")
ax1.set_xlabel("Time")
ax1.set_ylabel("Pitch (°)")
ax1.tick_params(axis='y')
ax1.legend(loc="upper left")
# Right axis: Roll
ax2 = ax1.twinx()
ax2.plot(df['Time'], df['Roll'], 'b-', label='Roll')
ax2.set_ylabel('Roll (°)', color='b')
ax2.tick_params(axis='y', labelcolor='b')
ax2.legend(loc="upper center")

# Right Y-axis for yaw
# ax3 = ax1.twinx()
# ax3.plot(df_filtered["Time"], df_filtered["Yaw"], label="Yaw (°)", color="tab:green", linestyle="dashed")
# ax3.set_ylabel("Yaw (°)")
# ax3.tick_params(axis='y')
# ax3.set_ylim(12, 16)
# ax3.legend(loc="upper right")

plt.title("Orientation (Pitch, Roll, Yaw) Over Time")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Plot acceleration
plt.figure(figsize=(10, 5))
plt.plot(df["Time"], df["Ax (g)"], label="Ax")
plt.plot(df["Time"], df["Ay (g)"], label="Ay")
plt.plot(df["Time"], df["Az (g)"], label="Az")
plt.title("Accelerometer Data Over Time")
plt.xlabel("Time")
plt.ylabel("Acceleration (g)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()